In [3]:
import re
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


# ============================================================================
# CONFIGURATION - SPECIFY LOG FILES TO ANALYZE
# ============================================================================


LOG_FILES_TO_ANALYZE = ['clam_patching.o513426',
'clam_patching.o513489',
'clam_patching.o513617',
'clam_patching.o513899',
'clam_patching.o514872',
'clam_patching.o515815',
'clam_patching.o515825',
'clam_patching_01.o515857',
'clam_patching_01.o515858',
'clam_patching_01.o515859',
'clam_patching_01.o515860',
'clam_patching_01.o515861',
'clam_patching_01.o515862',
'clam_patching_01.o515914',
'clam_patching_01_training_missing.o515899',
'MinicondaName.o1119570'
]


# ============================================================================
# PARSING HELPERS - COMPLETELY FIXED VERSION
# ============================================================================


def parse_log_entry(log_text):
    """
    Parse CLAM patching logs with proper WSI header + multiple contours approach.
    Returns a list of WSI-level dicts with aggregated contour data.
    """

    # Pattern for WSI header (everything before the first "Bounding Box:")
    wsi_header_pattern = (
        r'progress: ([\d.]+), (\d+)/(\d+)\n'
        r'processing (.+?\.tiff)\n'
        r'Seg level chosen: (\d+)\n'
        r'WSI Level dimensions: \((\d+), (\d+)\)\n'
        r'Creating patches for:\s+(.+?)\s+\.\.\.\n'
        r'Total number of contours to process:\s+(\d+)'
    )

    # Pattern for individual contour blocks
    contour_pattern = (
        r'Bounding Box: (\d+) (\d+) (\d+) (\d+)\n'
        r'Contour Area: ([\d.]+)\n'
        r'Extracted (\d+) coordinates'
    )

    # Split log into WSI sections (each starts with "progress:")
    wsi_sections = re.split(r'(?=progress: [\d.]+, \d+/\d+)', log_text)
    wsi_sections = [section.strip() for section in wsi_sections if section.strip()]

    wsi_entries = []

    for section in wsi_sections:
        # Extract WSI header info
        wsi_match = re.search(wsi_header_pattern, section, re.MULTILINE)
        if not wsi_match:
            continue

        # Extract all contour blocks from this WSI section
        contour_matches = re.findall(contour_pattern, section, re.MULTILINE)
        if not contour_matches:
            continue

        try:
            # Parse WSI header
            progress = float(wsi_match.group(1))
            current = int(wsi_match.group(2))
            total = int(wsi_match.group(3))
            filename = wsi_match.group(4)
            seg_level = int(wsi_match.group(5))
            wsi_width = int(wsi_match.group(6))
            wsi_height = int(wsi_match.group(7))
            sample_name = wsi_match.group(8)
            num_contours_declared = int(wsi_match.group(9))

            # Parse all contour blocks
            contours = []
            total_patches = 0
            total_contour_area = 0.0

            for contour_match in contour_matches:
                bbox_x = int(contour_match[0])
                bbox_y = int(contour_match[1])
                bbox_width = int(contour_match[2])
                bbox_height = int(contour_match[3])
                contour_area = float(contour_match[4])
                patches = int(contour_match[5])

                contours.append({
                    'bbox_x': bbox_x,
                    'bbox_y': bbox_y,
                    'bbox_width': bbox_width,
                    'bbox_height': bbox_height,
                    'contour_area': contour_area,
                    'patches': patches
                })

                total_patches += patches
                total_contour_area += contour_area

            # Create WSI entry with aggregated data
            wsi_entry = {
                'progress': progress,
                'current': current,
                'total': total,
                'filename': filename,
                'seg_level': seg_level,
                'wsi_width': wsi_width,
                'wsi_height': wsi_height,
                'sample_name': sample_name,
                'num_contours_declared': num_contours_declared,
                'num_contours': len(contours),  # Actual count from parsing
                'contours': contours,
                'total_patches': total_patches,
                'total_contour_area': total_contour_area
            }

            wsi_entries.append(wsi_entry)

        except (ValueError, IndexError) as e:
            print(f"Warning: Error parsing WSI section, skipping: {e}")
            continue

    return wsi_entries



def analyze_segmentation_quality(entry):
    """
    Compute quality metrics aggregating all contours per WSI.
    Always returns a dict.
    """
    w = entry.get('wsi_width', 0)
    h = entry.get('wsi_height', 0)
    wsi_total_pixels = int(w) * int(h) if w and h else 0


    total_patches = int(entry.get('total_patches', 0))
    total_contour_area = float(entry.get('total_contour_area', 0.0))


    # Use standard patch size 512x512
    patch_pixels = total_patches * 512 * 512


    # Combined bbox area across contours (simple sum - may have overlaps)
    contours = entry.get('contours', []) or []
    total_bbox_pixels = 0
    any_full_bbox = False

    for c in contours:
        bw = int(c.get('bbox_width', 0))
        bh = int(c.get('bbox_height', 0))
        bx = int(c.get('bbox_x', 0))
        by = int(c.get('bbox_y', 0))
        total_bbox_pixels += bw * bh

        # Check if this contour has full bbox (indicates segmentation failure)
        if w and h:
            if bx == 0 and by == 0 and bw == w and bh == h:
                any_full_bbox = True


    # Calculate coverage percentages
    contour_coverage = (total_contour_area / wsi_total_pixels) * 100 if wsi_total_pixels > 0 else 0.0
    patch_coverage = (patch_pixels / wsi_total_pixels) * 100 if wsi_total_pixels > 0 else 0.0
    bbox_coverage = (total_bbox_pixels / wsi_total_pixels) * 100 if wsi_total_pixels > 0 else 0.0
    patch_density_in_contour = (patch_pixels / total_contour_area) * 100 if total_contour_area > 0 else 0.0


    # Quality assessment rules
    if any_full_bbox and contour_coverage > 95:
        quality = "CRITICAL_FAIL"
        concern_level = 5
        reason = "Full bounding box + near-complete image coverage"
    elif any_full_bbox and contour_coverage > 90:
        quality = "FAILED"
        concern_level = 4
        reason = "Full bounding box + >90% contour coverage"
    elif any_full_bbox and contour_coverage > 70:
        quality = "HIGH_CONCERN"
        concern_level = 3
        reason = "Full bounding box + high contour coverage"
    elif contour_coverage > 95:
        quality = "SUSPICIOUS"
        concern_level = 2
        reason = "Very high contour coverage without full bbox"
    elif contour_coverage > 80 and patch_coverage > 40:
        quality = "MODERATE_CONCERN"
        concern_level = 1
        reason = "High contour and patch coverage"
    else:
        quality = "GOOD"
        concern_level = 0
        reason = "Normal segmentation pattern"


    # Patches per megapixel
    ppm = (total_patches / (wsi_total_pixels / 1_000_000.0)) if wsi_total_pixels > 0 else 0.0


    return {
        'quality': quality,
        'concern_level': concern_level,
        'reason': reason,
        'full_bbox': any_full_bbox,
        'contour_coverage': contour_coverage,
        'patch_coverage': patch_coverage,
        'bbox_coverage': bbox_coverage,
        'patch_density_in_contour': patch_density_in_contour,
        'wsi_total_pixels': wsi_total_pixels,
        'patch_pixels': patch_pixels,
        'patches_per_megapixel': ppm
    }



def extract_metadata(filename):
    """Extract institution and sample metadata from filename. Always returns dict."""
    basename = os.path.basename(filename) if filename else ""
    inst_match = re.match(r'(PIT_\d+)', basename)
    institution = inst_match.group(1) if inst_match else 'UNKNOWN'
    sample_match = re.search(r'PIT_\d+_(\d+)_\d+', basename)
    sample_id = sample_match.group(1) if sample_match else 'UNKNOWN'
    return {'institution': institution, 'sample_id': sample_id, 'basename': basename}


# ============================================================================
# MAIN ANALYSIS EXECUTION
# ============================================================================


print(" Selective CLAM Log File Analyzer - MULTI-CONTOUR FIXED VERSION")
print("=" * 60)


if not LOG_FILES_TO_ANALYZE:
    print("  WARNING: No log files specified!")
else:
    existing_files = []
    missing_files = []
    for log_file in LOG_FILES_TO_ANALYZE:
        if os.path.exists(log_file):
            existing_files.append(log_file)
        else:
            missing_files.append(log_file)


    print(f" Files to analyze: {len(LOG_FILES_TO_ANALYZE)}")
    print(f" Found: {len(existing_files)}")
    print(f" Missing: {len(missing_files)}")
    if missing_files:
        print(f"\nMissing files: {missing_files}")


    if not existing_files:
        print("\n No valid log files found to analyze!")
    else:
        all_entries = []
        file_stats = {}


        # Read and parse files
        for log_file in existing_files:
            print(f"\n Processing {log_file}...")
            try:
                with open(log_file, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()


                entries = parse_log_entry(content)  # Now properly parses WSI sections
                valid_entries = len(entries)
                if valid_entries > 0:
                    all_entries.extend(entries)
                    # Calculate total contours processed for this file
                    total_contours = sum(e['num_contours'] for e in entries)
                    total_patches = sum(e['total_patches'] for e in entries)
                    file_stats[log_file] = {
                        'valid_entries': valid_entries,
                        'total_contours': total_contours,
                        'total_patches': total_patches,
                        'file_size_mb': os.path.getsize(log_file) / (1024*1024)
                    }
                    print(f"    Found {valid_entries} WSI entries ({total_contours} total contours, {total_patches} total patches)")
                else:
                    print(f"     No valid bounding box entries found")
                    file_stats[log_file] = {'valid_entries': 0, 'file_size_mb': 0}
            except Exception as e:
                print(f"    Error reading {log_file}: {e}")
                file_stats[log_file] = {'error': str(e)}


        print(f"\n Total valid WSI entries found: {len(all_entries)}")


        if all_entries:
            # Remove duplicate WSIs by basename
            seen_files = set()
            unique_entries = []
            duplicates_removed = 0
            for entry in all_entries:
                filename = os.path.basename(entry.get('filename', ''))
                if filename and filename not in seen_files:
                    seen_files.add(filename)
                    unique_entries.append(entry)
                else:
                    duplicates_removed += 1


            print(f" Removed {duplicates_removed} duplicate entries")
            print(f" Unique WSIs to analyze: {len(unique_entries)}")


            # Show contour distribution statistics
            contour_counts = [e['num_contours'] for e in unique_entries]
            print(f" Contour distribution: Min={min(contour_counts)}, Max={max(contour_counts)}, Avg={np.mean(contour_counts):.1f}")
            print(f" Multi-contour WSIs: {sum(1 for c in contour_counts if c > 1)} ({sum(1 for c in contour_counts if c > 1)/len(contour_counts)*100:.1f}%)")


            # Analyze each unique WSI
            results = []
            for entry in unique_entries:
                analysis = analyze_segmentation_quality(entry)
                metadata = extract_metadata(entry.get('filename'))
                result = {**entry, **analysis, **metadata}
                results.append(result)


            # Build DataFrame
            df = pd.DataFrame(results)


            print("\n" + "=" * 80)
            print(" COMPREHENSIVE SEGMENTATION QUALITY ANALYSIS - MULTI-CONTOUR")
            print("=" * 80)


            # Summary statistics
            quality_counts = df['quality'].value_counts()
            concern_counts = df['concern_level'].value_counts().sort_index()
            total_images = len(df)


            print(f"\n OVERALL SUMMARY:")
            print(f"   Total unique images analyzed: {total_images:,}")
            print(f"\n QUALITY DISTRIBUTION:")
            for quality, count in quality_counts.items():
                percentage = (count / total_images) * 100
                emoji = {
                    "CRITICAL_FAIL": "🔴", "FAILED": "🔴", "HIGH_CONCERN": "🟠",
                    "SUSPICIOUS": "🟡", "MODERATE_CONCERN": "🟡", "GOOD": "🟢"
                }.get(quality, "⚪")
                print(f"   {emoji} {quality}: {count} ({percentage:.1f}%)")


            print(f"\n  CONCERN LEVEL BREAKDOWN:")
            concerning_files = 0
            for level in sorted(concern_counts.keys(), reverse=True):
                count = concern_counts[level]
                percentage = (count / total_images) * 100
                if level > 0:
                    concerning_files += count
                level_desc = {
                    5: " CRITICAL (Level 5)", 4: " FAILED (Level 4)",
                    3: " HIGH CONCERN (Level 3)", 2: " SUSPICIOUS (Level 2)",
                    1: " MODERATE (Level 1)", 0: " GOOD (Level 0)"
                }.get(level, f"Level {level}")
                print(f"   {level_desc}: {count} files ({percentage:.1f}%)")


            print(f"\n PIXEL USAGE STATISTICS:")
            print(f"   Average patch coverage: {df['patch_coverage'].mean():.1f}% ± {df['patch_coverage'].std():.1f}%")
            print(f"   Average contour coverage: {df['contour_coverage'].mean():.1f}% ± {df['contour_coverage'].std():.1f}%")
            print(f"   Files with >50% patch coverage: {(df['patch_coverage'] > 50).sum()} ({(df['patch_coverage'] > 50).mean() * 100:.1f}%)")
            print(f"   Files with >90% contour coverage: {(df['contour_coverage'] > 90).sum()} ({(df['contour_coverage'] > 90).mean() * 100:.1f}%)")


            # Institution analysis
            print(f"\n INSTITUTION ANALYSIS:")
            inst_analysis = df.groupby('institution').agg({
                'concern_level': ['count', 'mean', lambda x: (x > 0).sum()],
                'patch_coverage': 'mean',
                'contour_coverage': 'mean'
            }).round(2)
            inst_analysis.columns = ['total_files', 'avg_concern', 'concerning_files', 'avg_patch_cov', 'avg_contour_cov']
            inst_analysis['concern_rate'] = (inst_analysis['concerning_files'] / inst_analysis['total_files'] * 100).round(1)
            inst_analysis = inst_analysis.sort_values('concern_rate', ascending=False)


            for institution, row in inst_analysis.iterrows():
                print(f"   {institution}: {int(row['total_files'])} files, {int(row['concerning_files'])} concerning ({row['concern_rate']:.1f}%)")
                print(f"      Avg patch coverage: {row['avg_patch_cov']:.1f}%, Avg contour coverage: {row['avg_contour_cov']:.1f}%")


            # Export CSVs
            output_file = "comprehensive_segmentation_analysis_multicontour.csv"
            csv_columns = [
                'basename', 'institution', 'sample_id', 'quality', 'concern_level', 'reason',
                'wsi_width', 'wsi_height', 'total_patches', 'num_contours',
                'contour_coverage', 'patch_coverage', 'bbox_coverage', 'patch_density_in_contour',
                'patches_per_megapixel', 'full_bbox', 'wsi_total_pixels', 'patch_pixels'
            ]

            # Ensure missing columns exist
            for col in csv_columns:
                if col not in df.columns:
                    df[col] = np.nan


            df[csv_columns].to_csv(output_file, index=False)
            print(f"\n Comprehensive multi-contour analysis saved to {output_file}")


            # Concerning-only CSV
            concerning_df = df[df['concern_level'] > 0][csv_columns]
            concerning_output = "concerning_files_multicontour.csv"
            concerning_df.to_csv(concerning_output, index=False)
            print(f" Concerning files summary saved to {concerning_output}")


            # Final assessment
            print(f"\nFINAL ASSESSMENT:")
            good_files = (df['concern_level'] == 0).sum()
            good_percentage = (good_files / total_images) * 100 if total_images > 0 else 0
            print(f"Good quality files: {good_files} / {total_images} ({good_percentage:.1f}%)")
            print(f"Files needing attention: {concerning_files} / {total_images} ({(concerning_files/total_images)*100:.1f}%)")
            if total_images > 0:
                ratio = concerning_files / total_images
                if ratio < 0.05:
                    print("Dataset quality: EXCELLENT (< 5% concerning files)")
                elif ratio < 0.10:
                    print("Dataset quality: GOOD (< 10% concerning files)")
                elif ratio < 0.20:
                    print("Dataset quality: MODERATE (< 20% concerning files)")
                else:
                    print("Dataset quality: NEEDS ATTENTION (≥ 20% concerning files)")


            print(f"\n DataFrames available:")
            print(f"   'df' - Complete analysis ({len(df)} rows)")
            print(f"   'concerning_df' - Only concerning files ({len(concerning_df)} rows)")


            # ============================================================================
            # VISUALIZATION - SEPARATE PLOTS
            # ============================================================================

            # Plot 1: Quality distribution
            fig1, ax1 = plt.subplots(figsize=(10, 6))
            quality_counts.plot(kind='bar', ax=ax1, color=['red', 'orange', 'orange', 'yellow', 'yellow', 'green'])
            ax1.set_title('Segmentation Quality Distribution', fontsize=14, fontweight='bold')
            ax1.set_ylabel('Number of Files', fontsize=12)
            ax1.set_xlabel('Quality Category', fontsize=12)
            ax1.tick_params(axis='x', rotation=45)
            plt.tight_layout()
            plt.savefig('plot1_quality_distribution.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("\nSaved: plot1_quality_distribution.png")


            # Plot 2: Institution concern rates
            if len(inst_analysis) > 0:
                fig2, ax2 = plt.subplots(figsize=(10, 6))
                inst_analysis['concern_rate'].plot(kind='bar', ax=ax2, color='red', alpha=0.7)
                ax2.set_title('Concern Rate by Institution', fontsize=14, fontweight='bold')
                ax2.set_ylabel('Concern Rate (%)', fontsize=12)
                ax2.set_xlabel('Institution', fontsize=12)
                ax2.tick_params(axis='x', rotation=45)
                plt.tight_layout()
                plt.savefig('plot2_institution_concern_rates.png', dpi=300, bbox_inches='tight')
                plt.close()
                print("Saved: plot2_institution_concern_rates.png")


            # Plot 3: Patch coverage distribution
            fig3, ax3 = plt.subplots(figsize=(10, 6))
            mean_patch_coverage = df['patch_coverage'].mean()
            ax3.hist(df['patch_coverage'], bins=30, alpha=0.7, color='blue', edgecolor='black')
            ax3.axvline(mean_patch_coverage, color='red', linestyle='--', linewidth=2, 
                       label=f'Mean: {mean_patch_coverage:.1f}%')
            ax3.set_xlabel('Patch Coverage (%)', fontsize=12)
            ax3.set_ylabel('Number of Files', fontsize=12)
            ax3.set_title('Patch Coverage Distribution', fontsize=14, fontweight='bold')
            ax3.legend(fontsize=10)
            ax3.grid(alpha=0.3)
            plt.tight_layout()
            plt.savefig('plot3_patch_coverage_distribution.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("Saved: plot3_patch_coverage_distribution.png")


            # Plot 4: Contour vs Patch coverage scatter
            fig4, ax4 = plt.subplots(figsize=(10, 6))
            scatter = ax4.scatter(df['contour_coverage'], df['patch_coverage'],
                                 c=df['concern_level'], cmap='RdYlGn_r', alpha=0.6, s=50)
            ax4.set_xlabel('Contour Coverage (%)', fontsize=12)
            ax4.set_ylabel('Patch Coverage (%)', fontsize=12)
            ax4.set_title('Contour vs Patch Coverage', fontsize=14, fontweight='bold')
            ax4.grid(alpha=0.3)
            cbar = plt.colorbar(scatter, ax=ax4, label='Concern Level')
            cbar.set_label('Concern Level', fontsize=10)
            plt.tight_layout()
            plt.savefig('plot4_contour_vs_patch_coverage.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("Saved: plot4_contour_vs_patch_coverage.png")


            print("\nMulti-contour analysis complete!")
        else:
            print("\nNo valid entries found in any log files.")

 Selective CLAM Log File Analyzer - MULTI-CONTOUR FIXED VERSION
 Files to analyze: 16
 Found: 16
 Missing: 0

 Processing clam_patching.o513426...
    Found 885 WSI entries (4907 total contours, 768698 total patches)

 Processing clam_patching.o513489...
    Found 1004 WSI entries (5545 total contours, 764949 total patches)

 Processing clam_patching.o513617...
    Found 550 WSI entries (3176 total contours, 443923 total patches)

 Processing clam_patching.o513899...
    Found 926 WSI entries (4963 total contours, 727903 total patches)

 Processing clam_patching.o514872...
    Found 845 WSI entries (4849 total contours, 701308 total patches)

 Processing clam_patching.o515815...
    Found 807 WSI entries (4750 total contours, 719317 total patches)

 Processing clam_patching.o515825...
    Found 799 WSI entries (4676 total contours, 677042 total patches)

 Processing clam_patching_01.o515857...
    Found 14 WSI entries (108 total contours, 10024 total patches)

 Processing clam_patching